# MedSAM2 fixed 2D plane

One stationary 2D box, taken from the reference frame's 3D bounding box, and one fixed axial plane, the center slice of that same bounding box. MedSAM2 segments that plane on every frame with the same box prompt, so the contour is free to move or vanish as the lesion drifts through the plane.

In [1]:
%reload_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

# Notebook lives in CLI-Demos; `src` is one level up (engines/ceus).
sys.path.insert(0, str(Path.cwd().parent))
sys.path.insert(0, str(Path.cwd() / "medsam2_function"))

## Choose a visit

Run the first cell to list every row, then set `CASE_IDX`.

In [3]:
import pandas as pd

MASTER_CSV = "/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPUS/MedSAM2_TIC_results/MedSAM2_5fold_TIC_results.csv"

df = pd.read_csv(MASTER_CSV)
df[["Site", "Patient Number", "Visit", "Bolus"]]

,Site,Patient Number,Visit,Bolus
0,UCSD,P05,V01,CE1
1,UCSD,P05,V01,CE2
2,UCSD,P05,V02,CE1
3,UCSD,P05,V02,CE2
4,UCSD,P05,V03,CE1
5,UCSD,P05,V03,CE2
6,UCSD,P07,V01,CE1RUN2
7,UCSD,P07,V01,CE2
8,UCSD,P07,V02,CE01RUN2
9,UCSD,P07,V02,CE2


In [4]:
from case_files import find_bmode_file, find_ceus_file, find_voi_file

CASE_IDX = 39

row = df.iloc[CASE_IDX]
print(f"=== {row['Site']}-{row['Patient Number']}-{row['Visit']}-{row['Bolus']} ===")

bmode_path = find_bmode_file(row["Data Dir"], row["Visit"], row["Bolus"])
ceus_path = find_ceus_file(bmode_path)
voi_path = find_voi_file(row["Data Dir"], row["Visit"], row["Bolus"])
for name, p in (("bmode", bmode_path), ("ceus", ceus_path), ("voi", voi_path)):
    print(f"  {name:5s}: {os.path.basename(p)}")

=== SHC-P09-V01-CE2 ===
  bmode: SHC-P09-V01-CE2_18.45.33_mf_sip_capture_50_2_1_0_BMODE.nii
  ceus : SHC-P09-V01-CE2_18.45.33_mf_sip_capture_50_2_1_0_CEUS.nii
  voi  : SHC-P09-V01-CE2-MC_VOI.nii.gz


## Reference box and plane

The motion-compensation result is stored in the VOI file's header, so loading the VOI gives `seg_data.motion_compensation` with `reference_frame` and `reference_bbox` (the VOI's tight box plus the padding used when MC was run). The 2D prompt is that box's X/Y extent, and the plane is the middle of its Z extent. Set `Z_OVERRIDE` to use a different plane.

In [ ]:
from src.entrypoints import seg_loading_step

seg_data = seg_loading_step("nifti", None, voi_path, ceus_path)     # the NIfTI loader does not use image_data
mc = seg_data.motion_compensation
assert mc is not None, "This VOI file has no motion_compensation header; run motion compensation first."

ref_frame = mc.reference_frame
bb = mc.reference_bbox

Z_OVERRIDE = None
z = int((bb.z_min + bb.z_max) // 2) if Z_OVERRIDE is None else Z_OVERRIDE
box_2d = [int(v) for v in (bb.x_min, bb.y_min, bb.x_max, bb.y_max)]      # [x_min, y_min, x_max, y_max] on the (Y, X) plane

print(f"reference frame = {ref_frame}")
print(f"3D bbox  X=[{bb.x_min}, {bb.x_max}]  Y=[{bb.y_min}, {bb.y_max}]  Z=[{bb.z_min}, {bb.z_max}]")
print(f"fixed plane z = {z}   2D box = {box_2d}")

## Pull the axial video

The reference frame is always included.

In [ ]:
import numpy as np

from medsam2_axial_video import extract_axial_video

N_FRAMES = 100

import nibabel as nib

n_total = nib.load(bmode_path).shape[-1]
frame_indices = sorted(set(np.linspace(0, n_total - 1, N_FRAMES).round().astype(int).tolist()) | {ref_frame})

video = extract_axial_video(bmode_path, ceus_path, z, frame_indices=frame_indices)
assert seg_data.seg_mask.shape == nib.load(bmode_path).shape[:3], "VOI and B-mode are on different grids"
print(f"B-mode {video['bmode'].shape}  CEUS {video['ceus'].shape}  (T, H, W)")
print(f"frames {video['frame_indices']}")

## MedSAM2 on the fixed plane

Same fold-held-out checkpoint as the other notebooks. Each frame's B-mode plane gets the same `box_2d` prompt; there is no tracking and no use of the GT.

In [ ]:
import time

from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

from fold_checkpoints import checkpoint_for
from medsam2_viewer_base import run_medsam2_2d

model_cfg = "configs/sam2.1_hiera_t512.yaml"
checkpoint = os.environ.get("MEDSAM2_CHECKPOINT") or checkpoint_for(row["Site"], row["Patient Number"])
predictor = SAM2ImagePredictor(build_sam2(model_cfg, checkpoint, device="cuda"))

H, W = video["bmode"].shape[1:]
t0 = time.time()
masks = []
for plane in video["bmode"]:
    m = run_medsam2_2d(predictor, plane, box_2d, device="cuda")
    masks.append(np.zeros((H, W), bool) if m is None else m)
masks = np.stack(masks)
print(f"{len(masks)} frames in {time.time() - t0:.1f} s")
print("mask area (px):", masks.sum(axis=(1, 2)).tolist())

## MedSAM2 with the motion-compensated box

Same frames, checkpoint and **plane** (the fixed reference z: a sonographer's probe cannot follow the lesion in elevation), but the 2D box follows the lesion in-plane: on each frame it is the X/Y extent of that frame's **tracked** 3D box (`tracked_bboxes[t]`). This is the `sam2_2d_mc` arm. The fixed arm above keeps the reference box on every frame.


In [ ]:
tracked = mc.tracked_bboxes
frames_t = video["frame_indices"]

# plane stays at the reference z; only the box follows the tracked X/Y
boxes_mc = [[int(v) for v in (tracked[t].x_min, tracked[t].y_min, tracked[t].x_max, tracked[t].y_max)]
            for t in frames_t]

masks_mc = []
for plane, box in zip(video["bmode"], boxes_mc):
    m = run_medsam2_2d(predictor, plane, box, device="cuda")
    masks_mc.append(np.zeros((H, W), bool) if m is None else m)
masks_mc = np.stack(masks_mc)

print("z (both arms):", z)
print("mask area fixed box (px):", masks.sum(axis=(1, 2)).tolist())
print("mask area MC box    (px):", masks_mc.sum(axis=(1, 2)).tolist())


## Video: fixed box (top) vs motion-compensated box (bottom)

Each row is B-mode (left) and CEUS (right), both on the fixed plane `z`. Top row: the stationary reference box. Bottom row: the tracked box, which moves in-plane with the lesion. Cyan dashed = prompt box, lime = MedSAM2's mask, red = the drawn ROI (the VOI's slice at `z`, the same on every frame; it is a ground-truth outline only on the reference frame).

`DENOISE` is display only: `None` (raw, windowed to the `LOW_PCT`/`HIGH_PCT` percentiles of each stack), or `"none"`, `"nlm"`, `"wavelet"`, `"nlm_then_wavelet"` from `enhance_bmode_noise`, which denoises each frame and then does its own per-frame percentile stretch. MedSAM2 above always sees the raw B-mode.


In [ ]:
import contextlib
import io

import matplotlib.pyplot as plt
from IPython.display import HTML, display
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Rectangle

from medsam2_viewer_base import enhance_bmode_noise

DENOISE = "nlm"          # None | "none" | "nlm" | "wavelet" | "nlm_then_wavelet"
LOW_PCT, HIGH_PCT = 1, 99.5
INTERVAL_MS = 250

ref_pos = video["frame_indices"].index(ref_frame)
ref_plane = (seg_data.seg_mask[:, :, z] > 0).T                       # (Y, X)

def prepare(stack):
    """Return (frames, vmin, vmax) ready for imshow."""
    if DENOISE is None:
        lo, hi = np.percentile(stack, [LOW_PCT, HIGH_PCT])
        return stack, lo, hi
    with contextlib.redirect_stdout(io.StringIO()):      # drop the per-frame timing prints
        out = np.stack([enhance_bmode_noise(f, LOW_PCT, HIGH_PCT, method=DENOISE) for f in stack])
    return out, 0, 255

# row -> (label, masks, per-frame boxes); both rows show the same fixed-plane video
rows = {
    "fixed": ("Fixed box", masks, [box_2d] * len(masks)),
    "mc":    ("MC box", masks_mc, boxes_mc),
}
prepared_video = {m: prepare(video[m]) for m in ("bmode", "ceus")}

fig, axes = plt.subplots(2, 2, figsize=(11, 9))
ims, rects = {}, {}
for r, (key, (label, _, boxes)) in enumerate(rows.items()):
    for c, (mod, name) in enumerate((("bmode", "B-mode"), ("ceus", "CEUS"))):
        ax = axes[r, c]
        frames, lo, hi = prepared_video[mod]
        ims[key, mod] = ax.imshow(frames[0], cmap="gray", vmin=lo, vmax=hi)
        x0, y0, x1, y1 = boxes[0]
        rects[key, mod] = ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, ec="cyan", ls="--", lw=1))
        ax.axis("off")
title = fig.suptitle("")
contours = []

def update(i):
    for cs in contours:
        cs.remove()
    contours.clear()
    for r, (key, (label, msk, boxes)) in enumerate(rows.items()):
        x0, y0, x1, y1 = boxes[i]
        for c, (mod, name) in enumerate((("bmode", "B-mode"), ("ceus", "CEUS"))):
            ax = axes[r, c]
            ims[key, mod].set_data(prepared_video[mod][0][i])
            rects[key, mod].set_bounds(x0, y0, x1 - x0, y1 - y0)
            for mask, color in ((msk[i], "lime"), (ref_plane, "red")):
                if mask.any():
                    contours.append(ax.contour(mask, levels=[0.5], colors=color, linewidths=1.2))
            ax.set_title(f"{name} - {label}  z={z}  area={int(msk[i].sum())} px", fontsize=10)
    title.set_text(f"t={video['frame_indices'][i]}  denoise={DENOISE}" + ("   [reference frame]" if i == ref_pos else ""))
    return list(ims.values())

anim = FuncAnimation(fig, update, frames=len(video["frame_indices"]), interval=INTERVAL_MS, blit=False)
plt.close(fig)
display(HTML(anim.to_jshtml()))

## TIC on the fixed plane (2D only)

Three ROIs on the same fixed plane `z`: the drawn ROI (the VOI's slice at `z`, unchanged over time), MedSAM2 with the stationary reference box (`sam2_2d_fixed`), and MedSAM2 with the tracked X/Y box (`sam2_2d_mc`). The TIC is the framework's `tic` curve (`CurvesAnalysis`, via `medsam2_tic.py`) and the log-normal fit is `curve_quantification_step` with `lognormal_fit_full`, the same functions as the 3D TIC cells in MedSam2_inference. The framework assumes consecutive frames, so this pulls and segments **every** frame, not the `N_FRAMES` subset shown above.


In [ ]:
from tqdm import tqdm

from src.data_objs.image import UltrasoundImage
from src.data_objs.seg import CeusSeg

from medsam2_tic import fit_quality, fitted_curve, lognormal_quantification, tic_analysis

full = extract_axial_video(bmode_path, ceus_path, z, frame_indices=list(range(n_total)))

masks_full = []
mask_mc = []
for plane in tqdm(full["bmode"], desc="MedSAM2 [fixed plane]"):
    m = run_medsam2_2d(predictor, plane, box_2d, device="cuda")
    masks_full.append(np.zeros((H, W), bool) if m is None else m)
masks_full = np.stack(masks_full)

# (T, H, W) CEUS planes as the framework's "2D + time" image, frame rate as the NIfTI loader sets it
image_2d = UltrasoundImage(ceus_path)
image_2d.pixel_data = image_2d.intensities_for_analysis = full["ceus"]
image_2d.pixdim = full["spacing_mm_yx"]
image_2d.frame_rate = 1.0 / nib.load(ceus_path).header.get_zooms()[3]
seg_2d = CeusSeg()
seg_2d.seg_name = "medsam2_fixed_plane"

analysis_2d = tic_analysis(image_2d, seg_2d, lambda f: masks_full[f], desc="TIC [fixed plane]")
tic_2d = np.asarray(analysis_2d.curves[0]["TIC"], dtype=float)

# drawn ROI: the VOI's slice at z, the same mask on every frame
analysis_2d_roi = tic_analysis(image_2d, seg_2d, lambda f: ref_plane, desc="TIC [drawn ROI]")
tic_2d_roi = np.asarray(analysis_2d_roi.curves[0]["TIC"], dtype=float)
print(f"{len(tic_2d)} frames  min {np.nanmin(tic_2d):.2f}  max {np.nanmax(tic_2d):.2f}  empty-mask frames: {int((masks_full.sum(axis=(1, 2)) == 0).sum())}")

In [ ]:
# ── TIC with the motion-compensated box (same fixed plane) ───────────────────
tracked = mc.tracked_bboxes
assert len(tracked) == n_total, "expected one tracked box per frame"

boxes_mc_all = [[int(v) for v in (bb.x_min, bb.y_min, bb.x_max, bb.y_max)] for bb in tracked]

masks_mc_all = []
for plane, box in tqdm(zip(full["bmode"], boxes_mc_all), total=n_total, desc="MedSAM2 [MC box]"):
    m = run_medsam2_2d(predictor, plane, box, device="cuda")
    masks_mc_all.append(np.zeros((H, W), bool) if m is None else m)
masks_mc_all = np.stack(masks_mc_all)

analysis_2d_mc = tic_analysis(image_2d, seg_2d, lambda f: masks_mc_all[f], desc="TIC [MC box]")
quant_2d_mc = lognormal_quantification(analysis_2d_mc)
fit_2d_mc = fitted_curve(analysis_2d_mc, quant_2d_mc)
tic_2d_mc = np.asarray(analysis_2d_mc.curves[0]["TIC"], dtype=float)


In [ ]:
quant_2d = lognormal_quantification(analysis_2d)
fit_2d = fitted_curve(analysis_2d, quant_2d)
quant_2d_roi = lognormal_quantification(analysis_2d_roi)
fit_2d_roi = fitted_curve(analysis_2d_roi, quant_2d_roi)
time_2d = analysis_2d.time_arr

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(time_2d, tic_2d_roi, color="green", linewidth=2, alpha=0.4, label="Drawn ROI raw TIC")
ax.plot(time_2d, fit_2d_roi, color="darkgreen", linewidth=2, linestyle="--", label="Drawn ROI fitted")
ax.plot(time_2d, tic_2d, color="blue", linewidth=2, alpha=0.4, label="MedSAM2 fixed-box raw TIC")
ax.plot(time_2d, fit_2d, color="darkblue", linewidth=2, linestyle="--", label="MedSAM2 fixed-box fitted")
ax.plot(time_2d, tic_2d_mc, color="red", linewidth=2, alpha=0.4, label="MedSAM2 MC-box raw TIC")
ax.plot(time_2d, fit_2d_mc, color="darkred", linewidth=2, linestyle="--", label="MedSAM2 MC-box fitted")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Mean intensity")
ax.set_title(f"2D TIC at z={z}")
ax.legend(loc="upper right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

quality = {name: fit_quality(a, q) for name, a, q in (("drawn ROI", analysis_2d_roi, quant_2d_roi),
                                                      ("sam2 fixed", analysis_2d, quant_2d),
                                                      ("sam2 MC", analysis_2d_mc, quant_2d_mc))}
print(f"{'':<10}" + "".join(f"{name:>14}" for name in quality))
for key in quality["drawn ROI"]:
    print(f"{key:<10}" + "".join(f"{q[key]:>14.4f}" for q in quality.values()))
